In [6]:
"""
MGIC — real-image demo
======================
Encode a photograph as a finite moving Gaussian measure:

    I(x) = sum_k w_k N(x; mu_k, Sigma_k) c_k

Pipeline:
    1. density  p(x)  from luminance
    2. weighted EM  ->  (w, mu, Sigma)
    3. ridge LSQ    ->  c
    4. gradient flow on (mu, Sigma, c, logits_w)  with cosine LR

Deps: numpy, torch, matplotlib, pillow  (+ scikit-image or scipy recommended)
"""

import io, os, math, time, urllib.request
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from PIL import Image, ImageFilter

np.random.seed(0)
torch.manual_seed(0)

# ----------------------------------------------------------------------
# 0.  Real image loading
# ----------------------------------------------------------------------

_URLS = [
    "https://upload.wikimedia.org/wikipedia/commons/thumb/2/2a/"
    "Eiffel_Tower_from_Champ_de_Mars%2C_Paris%2C_France.jpg/"
    "512px-Eiffel_Tower_from_Champ_de_Mars%2C_Paris%2C_France.jpg",
    "https://upload.wikimedia.org/wikipedia/commons/thumb/8/8e/"
    "Cat_November_2010-1a.jpg/512px-Cat_November_2010-1a.jpg",
]


def _try_local_libs():

    try:
        from scipy.datasets import face
        return face(), "scipy.face"
    except Exception:
        pass
    try:
        from scipy.misc import face
        return face(), "scipy.face"
    except Exception:
        pass
    return None, None


def _try_download():
    for url in _URLS:
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=15) as r:
                buf = io.BytesIO(r.read())
            arr = np.asarray(Image.open(buf).convert("RGB"), np.uint8)
            return arr, f"url({os.path.basename(url)[:24]}...)"
        except Exception:
            continue
    return None, None


def load_image(size=128):
    arr, src = _try_local_libs()
    if arr is None:
        arr, src = _try_download()
    if arr is None:
        # last-resort synthetic
        n = 512
        y, x = np.mgrid[0:n, 0:n] / n
        r = np.sqrt((x - 0.5) ** 2 + (y - 0.5) ** 2)
        arr = np.zeros((n, n, 3), np.uint8)
        arr[..., 0] = np.clip(180 + 60 * np.sin(22 * x) * np.cos(7 * y), 0, 255)
        arr[..., 1] = np.clip(180 + 60 * np.cos(18 * y), 0, 255)
        arr[..., 2] = np.clip(180 + 60 * np.sin(14 * (x + y)), 0, 255)
        arr *= (r < 0.48)[..., None]
        src = "synthetic"

    if arr.ndim == 2:
        arr = np.stack([arr] * 3, -1)
    img = Image.fromarray(arr.astype(np.uint8)).convert("RGB")
    img = img.resize((size, size), Image.LANCZOS)
    return np.asarray(img).astype(np.float32) / 255.0, src


# ----------------------------------------------------------------------
# 1.  Metrics and DCT
# ----------------------------------------------------------------------

def psnr(a, b):
    mse = float(np.mean((a - b) ** 2))
    return 10.0 * np.log10(1.0 / max(mse, 1e-12))


def _box(x, r):
    p = np.pad(x, r, mode="edge")
    c = np.cumsum(np.cumsum(p, 0), 1)
    c = np.pad(c, ((1, 0), (1, 0)))
    H, W = x.shape
    s = (c[2 * r + 1:2 * r + 1 + H, 2 * r + 1:2 * r + 1 + W]
         - c[0:H, 2 * r + 1:2 * r + 1 + W]
         - c[2 * r + 1:2 * r + 1 + H, 0:W]
         + c[0:H, 0:W])
    return s / ((2 * r + 1) ** 2)


def ssim(a, b, r=5):
    C1, C2 = 0.01 ** 2, 0.03 ** 2
    vals = []
    for ch in range(a.shape[2]):
        x, y = a[..., ch], b[..., ch]
        mx, my = _box(x, r), _box(y, r)
        vx = _box(x * x, r) - mx * mx
        vy = _box(y * y, r) - my * my
        vxy = _box(x * y, r) - mx * my
        s = ((2 * mx * my + C1) * (2 * vxy + C2)) / ((mx * mx + my * my + C1) * (vx + vy + C2))
        vals.append(float(s.mean()))
    return float(np.mean(vals))


def dct_matrix(N):
    n = np.arange(N)
    k = np.arange(N)[:, None]
    D = np.cos(np.pi * (2 * n + 1) * k / (2 * N)) * np.sqrt(2.0 / N)
    D[0] /= np.sqrt(2.0)
    return D


def dct2(X):
    D = dct_matrix(X.shape[0])
    return D @ X @ D.T


def idct2(X):
    D = dct_matrix(X.shape[0])
    return D.T @ X @ D


# ----------------------------------------------------------------------
# 2.  Weighted EM for 2-D Gaussian mixture
# ----------------------------------------------------------------------

def _sqdist(X, mu):
    return (X ** 2).sum(1)[:, None] - 2 * X @ mu.T + (mu ** 2).sum(1)[None, :]


def fit_gmm_em(X, p, K, iters=60, reg=1e-5, seed=0, verbose=True):
    rng = np.random.default_rng(seed)
    N, D = X.shape

    idx = rng.choice(N, size=K, p=p, replace=False)
    mu = X[idx].copy()
    for _ in range(8):
        lab = _sqdist(X, mu).argmin(1)
        for k in range(K):
            m = lab == k
            ws = p[m].sum()
            if ws > 1e-12:
                mu[k] = (p[m, None] * X[m]).sum(0) / ws

    Sigma = np.tile(np.eye(D) * (0.5 / np.sqrt(K)) ** 2, (K, 1, 1))
    w = np.full(K, 1.0 / K)
    log2pi = D * np.log(2 * np.pi)
    prev_ll = -np.inf

    for it in range(iters):
        logN = np.empty((N, K), np.float64)
        for k in range(K):
            L = np.linalg.cholesky(Sigma[k])
            z = np.linalg.solve(L, (X - mu[k]).T)
            quad = (z ** 2).sum(0)
            logdet = 2.0 * np.log(np.diag(L)).sum()
            logN[:, k] = -0.5 * (quad + logdet + log2pi)

        logpost = logN + np.log(w + 1e-300)[None, :]
        m = logpost.max(1, keepdims=True)
        e = np.exp(logpost - m)
        gamma = e / e.sum(1, keepdims=True)
        r = p[:, None] * gamma
        Nk = r.sum(0) + 1e-12

        w = Nk / Nk.sum()
        mu = (r.T @ X) / Nk[:, None]
        for k in range(K):
            d = X - mu[k]
            Sigma[k] = (r[:, k, None] * d).T @ d / Nk[k] + reg * np.eye(D)

        ll = float((p * np.log(np.maximum((np.exp(logN) * w[None, :]).sum(1), 1e-300))).sum())
        if verbose and (it % 15 == 0 or it == iters - 1):
            print(f"    EM {it:3d}   loglik = {ll: .5f}")
        if abs(ll - prev_ll) < 1e-8:
            break
        prev_ll = ll

    return w, mu, Sigma


def gaussian_basis(X, w, mu, Sigma):
    N, K = X.shape[0], w.shape[0]
    A = np.empty((N, K), np.float32)
    for k in range(K):
        L = np.linalg.cholesky(Sigma[k])
        z = np.linalg.solve(L, (X - mu[k]).T)
        quad = (z ** 2).sum(0)
        logdet = 2.0 * np.log(np.diag(L)).sum()
        A[:, k] = (w[k] * np.exp(-0.5 * (quad + logdet + 2 * np.log(2 * np.pi)))).astype(np.float32)
    return A


def fit_colors(A, I, ridge=1e-3):
    K = A.shape[1]
    G = A.T @ A + ridge * np.eye(K, dtype=np.float32)
    b = A.T @ I
    return np.linalg.solve(G, b).astype(np.float32)


# ----------------------------------------------------------------------
# 3.  Differentiable atom model
# ----------------------------------------------------------------------

class MGIC(torch.nn.Module):
    """
    I(x) = sum_k w_k N(x; mu_k, Sigma_k) c_k
    Parameters: mu (K,2), Lraw (K,2,2) with Sigma = L L^T, c (K,C),
                logits_w (K,) with w = softmax(logits_w) (optional).
    """

    def __init__(self, w, mu, Sigma, c, learn_w=False):
        super().__init__()
        K = w.shape[0]
        w_log = np.log(np.maximum(w, 1e-12))
        w_log = w_log - w_log.mean()
        self.logits_w = torch.nn.Parameter(torch.as_tensor(w_log, dtype=torch.float32))
        self.learn_w = learn_w
        self.register_buffer("w0", torch.as_tensor(w, dtype=torch.float32))

        self.mu = torch.nn.Parameter(torch.as_tensor(mu, dtype=torch.float32))

        L = np.linalg.cholesky(Sigma).astype(np.float32)
        Lraw = L.copy()
        Lraw[:, 0, 0] = np.log(L[:, 0, 0])
        Lraw[:, 1, 1] = np.log(L[:, 1, 1])
        self.Lraw = torch.nn.Parameter(torch.as_tensor(Lraw))
        self.c = torch.nn.Parameter(torch.as_tensor(c, dtype=torch.float32))
        self.K = K

    def weights(self):
        if self.learn_w:
            return torch.softmax(self.logits_w, 0)
        return self.w0

    def _L(self):
        L = torch.tril(self.Lraw, -1)
        d = torch.diagonal(self.Lraw, dim1=-2, dim2=-1)
        return L + torch.diag_embed(torch.exp(d))

    def forward(self, grid):
        L = self._L()
        Sigma = L @ L.transpose(-1, -2)
        P = torch.linalg.inv(Sigma)
        logdet = 2.0 * torch.log(torch.diagonal(L, dim1=-2, dim2=-1)).sum(-1)

        P00, P01, P11 = P[:, 0, 0], P[:, 0, 1], P[:, 1, 1]
        Pm = torch.einsum("kij,kj->ki", P, self.mu)
        mPm = (self.mu * Pm).sum(-1)

        x = grid[:, 0][:, None]
        y = grid[:, 1][:, None]
        xPx = P00[None, :] * x * x + 2.0 * P01[None, :] * x * y + P11[None, :] * y * y
        xPm = grid @ Pm.T
        quad = xPx - 2.0 * xPm + mPm[None, :]

        logN = -0.5 * (quad + logdet[None, :] + 2.0 * np.log(2 * np.pi))
        basis = torch.exp(logN) * self.weights()[None, :]
        return basis @ self.c


def refine(model, target, grid, steps=500, lr=1.2e-2, verbose=True):
    params = [
        {"params": [model.mu], "lr": lr},
        {"params": [model.Lraw], "lr": lr * 0.5},
        {"params": [model.c], "lr": lr},
    ]
    if model.learn_w:
        params.append({"params": [model.logits_w], "lr": lr})
    opt = torch.optim.Adam(params)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps, eta_min=lr * 0.05)

    for s in range(steps):
        opt.zero_grad()
        out = model(grid)
        loss = F.mse_loss(out, target)
        loss.backward()
        opt.step()
        sched.step()
        if verbose and (s % 50 == 0 or s == steps - 1):
            print(f"    flow {s:4d}   mse = {loss.item():.6f}   lr = {sched.get_last_lr()[0]:.2e}")
    return model


# ----------------------------------------------------------------------
# 4.  Baselines
# ----------------------------------------------------------------------

def baseline_bilinear(img, budget):
    H, W, C = img.shape
    s = max(1, int(round(math.sqrt(C * H * W / max(budget, 1)))))
    small = Image.fromarray((img * 255).astype(np.uint8)).resize((W // s, H // s), Image.BILINEAR)
    up = Image.fromarray(np.asarray(small)).resize((W, H), Image.BILINEAR)
    return np.asarray(up).astype(np.float32) / 255.0, (W // s) * (H // s) * C, f"bilinear 1/{s}"


def baseline_dct(img, budget):
    H, W, C = img.shape
    M = max(1, budget // C)
    out = np.zeros_like(img)
    used = 0
    for ch in range(C):
        D = dct2(img[..., ch])
        flat = np.abs(D).ravel()
        thr = np.partition(flat, -M)[-M]
        mask = np.abs(D) >= thr
        used += int(mask.sum())
        out[..., ch] = idct2(D * mask)
    return np.clip(out, 0, 1), used, f"DCT top-{M}/ch"


def baseline_jpeg(img, quality):
    buf = io.BytesIO()
    Image.fromarray((img * 255).astype(np.uint8)).save(buf, "JPEG", quality=quality)
    nbytes = buf.tell()
    buf.seek(0)
    out = np.asarray(Image.open(buf).convert("RGB")).astype(np.float32) / 255.0
    return out, nbytes, f"JPEG q={quality}"


def baseline_blur(img, sigma_pix):
    out = Image.fromarray((img * 255).astype(np.uint8)).filter(ImageFilter.GaussianBlur(sigma_pix))
    return np.asarray(out).astype(np.float32) / 255.0, 0, f"blur σ={sigma_pix}px"


# ----------------------------------------------------------------------
# 5.  Atom visualization
# ----------------------------------------------------------------------

def draw_atoms(ax, mu, Sigma, w, c):
    ax.set_facecolor("black")
    ax.set_xlim(0, 1)
    ax.set_ylim(1, 0)
    ax.set_aspect("equal")
    wmax = w.max() if w.max() > 0 else 1.0
    for k in np.argsort(w):
        vals, vecs = np.linalg.eigh(Sigma[k])
        vals = np.clip(vals, 1e-9, None)
        ang = math.degrees(math.atan2(vecs[1, 0], vecs[0, 0]))
        wdt, hgt = 2.0 * math.sqrt(vals[0]), 2.0 * math.sqrt(vals[1])
        col = np.clip(c[k], 0, 1)
        alpha = float(np.clip(0.15 + 0.85 * (w[k] / wmax) ** 0.5, 0.05, 0.95))
        ax.add_patch(Ellipse(mu[k], wdt, hgt, angle=ang,
                             facecolor=col, edgecolor="none", alpha=alpha))
    ax.set_xticks([])
    ax.set_yticks([])


# ----------------------------------------------------------------------
# 6.  Main
# ----------------------------------------------------------------------

def main(H=128, K=512, EM_ITERS=60, FLOW_STEPS=500, LEARN_W=True):
    W, C = H, 3
    N = H * W

    img, src = load_image(H)
    print(f"source image : {src}")
    print(f"image {H}x{W}x{C}   atoms K={K}   grid pixels N={N}")
    target_np = img.reshape(-1, C)

    yy, xx = np.mgrid[0:H, 0:W]
    grid_np = np.stack([(xx.ravel() + 0.5) / W, (yy.ravel() + 0.5) / H], -1).astype(np.float64)
    grid = torch.tensor(grid_np, dtype=torch.float32)
    target = torch.tensor(target_np)

    # density from luminance (plus a floor so dark regions still attract atoms)
    lum = img.mean(-1).ravel().astype(np.float64)
    p = lum + 0.30 * lum.mean()
    p = p / p.sum()

    # EM
    print("\n[1] fitting Gaussian atoms by weighted EM")
    t0 = time.time()
    w, mu, Sigma = fit_gmm_em(grid_np, p, K, iters=EM_ITERS)
    print(f"    done in {time.time() - t0:.1f}s")

    # appearance
    print("\n[2] solving for atom appearances (ridge LSQ)")
    A = gaussian_basis(grid_np.astype(np.float32), w.astype(np.float32),
                       mu.astype(np.float32), Sigma.astype(np.float32))
    c = fit_colors(A, target_np)
    rec_em = np.clip((A @ c).reshape(H, W, C), 0, 1)

    # flow
    print(f"\n[3] gradient flow on (mu, Sigma, c{' , logits_w' if LEARN_W else ''})")
    model = MGIC(w, mu, Sigma, c, learn_w=LEARN_W)
    t0 = time.time()
    refine(model, target, grid, steps=FLOW_STEPS, lr=1.2e-2)
    print(f"    done in {time.time() - t0:.1f}s")

    with torch.no_grad():
        rec_flow = np.clip(model(grid).numpy().reshape(H, W, C), 0, 1)
        L = model._L().numpy()
        Sigma_r = L @ L.transpose(0, 2, 1)
        mu_r = model.mu.numpy()
        c_r = model.c.numpy()
        w_r = model.weights().numpy()

    mgic_params = 9 * K
    print(f"\natom budget: {K} atoms × 9 params = {mgic_params} values "
          f"(raw grid = {N * C})")

    # baselines
    print("\n[4] baselines")
    b_bil, n_bil, name_bil = baseline_bilinear(img, mgic_params)
    b_dct, n_dct, name_dct = baseline_dct(img, mgic_params)
    b_jpg, n_jpg, name_jpg = baseline_jpeg(img, 20)
    b_blr, _, name_blr = baseline_blur(img, 2.0)

    rows = [
        ("original",         img,      N * C,       None),
        ("MGIC (EM only)",   rec_em,   mgic_params, None),
        ("MGIC (EM + flow)", rec_flow, mgic_params, None),
        (name_bil,           b_bil,    n_bil,       None),
        (name_dct,           b_dct,    n_dct,       None),
        (name_jpg,           b_jpg,    n_jpg,       "bytes"),
        (name_blr,           b_blr,    0,           "not a codec"),
    ]

    print("\n" + "=" * 74)
    print(f"{'method':<22}{'params':>12}{'PSNR (dB)':>12}{'SSIM':>10}")
    print("-" * 74)
    for name, rec, nparam, note in rows:
        if name == "original":
            print(f"{name:<22}{nparam:>12}{'--':>12}{'1.000':>10}")
            continue
        p_ = psnr(rec, img)
        s_ = ssim(rec, img)
        tag = f" ({note})" if note else ""
        print(f"{(name + tag):<22}{nparam:>12}{p_:>12.2f}{s_:>10.4f}")
    print("=" * 74)

    # per-channel
    print("\nper-channel PSNR (refined MGIC vs. original)")
    for i, ch in enumerate("RGB"):
        print(f"  {ch}: {psnr(rec_flow[..., i], img[..., i]):5.2f} dB   "
              f"mean |c_{ch}| = {np.abs(c_r[:, i]).mean():.3f}")

    # figure
    fig, axes = plt.subplots(2, 4, figsize=(15, 7.6))
    panels = [
        (axes[0, 0], img,      "original"),
        (axes[0, 1], rec_em,   f"MGIC EM only   {psnr(rec_em, img):.2f} dB"),
        (axes[0, 2], rec_flow, f"MGIC EM+flow   {psnr(rec_flow, img):.2f} dB"),
        (None,       None,     f"atoms  K={K}"),
        (axes[1, 0], b_bil,    f"{name_bil}   {psnr(b_bil, img):.2f} dB"),
        (axes[1, 1], b_dct,    f"{name_dct}   {psnr(b_dct, img):.2f} dB"),
        (axes[1, 2], b_jpg,    f"{name_jpg}   {psnr(b_jpg, img):.2f} dB"),
        (axes[1, 3], b_blr,    f"{name_blr}   {psnr(b_blr, img):.2f} dB"),
    ]
    for ax, im, title in panels:
        if ax is None:
            ax = axes[0, 3]
            draw_atoms(ax, mu_r, Sigma_r, w_r, c_r)
        else:
            ax.imshow(im)
            ax.set_xticks([])
            ax.set_yticks([])
        ax.set_title(title, fontsize=10)

    fig.suptitle(
        f"MGIC: {K} moving Gaussian atoms on {src}   "
        f"({H}×{W}×{C}, atom budget {mgic_params} values)",
        fontsize=12,
    )
    fig.tight_layout()
    fig.savefig("mgic_real.png", dpi=130, bbox_inches="tight")
    print("\nsaved  mgic_real.png")

    # atom statistics
    print("\natom statistics (refined)")
    tr = np.trace(Sigma_r, axis1=1, axis2=2)
    print(f"  weight   min/med/max : {w_r.min():.3e} / {np.median(w_r):.3e} / {w_r.max():.3e}")
    print(f"  trace Σ  min/med/max : {tr.min():.3e} / {np.median(tr):.3e} / {tr.max():.3e}")
    print(f"  |c|      med         : {np.median(np.abs(c_r)):.3f}")
    eff = 1.0 / np.sum((w_r / w_r.sum()) ** 2)
    print(f"  effective #atoms 1/Σw² : {eff:.1f} of {K}")

    return img, rec_em, rec_flow, (mu_r, Sigma_r, w_r, c_r)


if __name__ == "__main__":
    main()

source image : synthetic
image 128x128x3   atoms K=512   grid pixels N=16384

[1] fitting Gaussian atoms by weighted EM
    EM   0   loglik =  0.12518
    EM  15   loglik =  0.13888
    EM  30   loglik =  0.14003
    EM  45   loglik =  0.14096
    EM  59   loglik =  0.14150
    done in 33.3s

[2] solving for atom appearances (ridge LSQ)

[3] gradient flow on (mu, Sigma, c , logits_w)
    flow    0   mse = 0.002339   lr = 1.20e-02
    flow   50   mse = 0.001202   lr = 1.17e-02
    flow  100   mse = 0.000697   lr = 1.09e-02
    flow  150   mse = 0.000498   lr = 9.62e-03
    flow  200   mse = 0.000364   lr = 8.03e-03
    flow  250   mse = 0.000284   lr = 6.26e-03
    flow  300   mse = 0.000232   lr = 4.50e-03
    flow  350   mse = 0.000204   lr = 2.92e-03
    flow  400   mse = 0.000188   lr = 1.67e-03
    flow  450   mse = 0.000178   lr = 8.68e-04
    flow  499   mse = 0.000172   lr = 6.00e-04
    done in 36.8s

atom budget: 512 atoms × 9 params = 4608 values (raw grid = 49152)

[4] basel

In [7]:
#!/usr/bin/env python3
"""
GLUE sanity check — freeze-depth overparameterization at fine-tune time
======================================================================

Single-file, Kaggle T4, no PEFT / no HuggingFace Trainer.

Grid (default --preset quick)
-----------------------------
  Models : bert-base-uncased, roberta-base
  Tasks  : SST-2 (easy) and RTE (small-n / harder)
  Freeze : head_only  |  last2 encoder blocks  |  full_ft
  Seed   : 42

SST-2 is capped at 8k train examples in QUICK so the 2×3 freeze grid
fits a T4. RTE uses the full 2,490. Use --preset full for uncropped SST-2.

Kaggle
------
  Internet ON, GPU T4.
  !pip install -q "transformers>=4.36" datasets scikit-learn pandas matplotlib
  !python glue_dd_sanity.py --preset quick

  Smoke (~8 min):  python glue_dd_sanity.py --preset smoke

Notebook-safe: unknown argv (Jupyter -f kernel.json) is ignored.
"""

from __future__ import annotations

import argparse
import json
import math
import os
import random
import subprocess
import sys
import time
import warnings
from pathlib import Path
from typing import Any

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")


def _ensure_pkgs() -> None:
    missing = []
    for pkg, name in [
        ("transformers", "transformers"),
        ("datasets", "datasets"),
        ("scikit-learn", "sklearn"),
        ("pandas", "pandas"),
        ("matplotlib", "matplotlib"),
    ]:
        try:
            __import__(name)
        except ImportError:
            missing.append(pkg)
    if missing:
        print(f"[setup] pip install {missing}", flush=True)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])


_ensure_pkgs()

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# ---------------------------------------------------------------------------
# Catalog
# ---------------------------------------------------------------------------

TASKS = {
    "sst2": {
        "display": "SST-2  (easy)",
        "s1": "sentence",
        "s2": None,
        "metric": "accuracy",
        "metric_label": "Accuracy",
        "n_train": 67349,
        "n_val": 872,
        "kind": "easy",
    },
    "rte": {
        "display": "RTE  (complex / small-n)",
        "s1": "sentence1",
        "s2": "sentence2",
        "metric": "accuracy",
        "metric_label": "Accuracy",
        "n_train": 2490,
        "n_val": 277,
        "kind": "complex",
    },
}

MODELS = {
    "bert-base-uncased": {
        "display": "BERT-base",
        "family": "bert",
        "encoder_attr": ("bert", "encoder", "layer"),
        "pooler_attr": ("bert", "pooler"),
        "n_layers": 12,
        "color": "#1f4e79",
    },
    "roberta-base": {
        "display": "RoBERTa-base",
        "family": "roberta",
        "encoder_attr": ("roberta", "encoder", "layer"),
        "pooler_attr": None,  # RobertaClassificationHead, no bert-style pooler
        "n_layers": 12,
        "color": "#b85c38",
    },
}

# Paper TEST = Devlin / Liu official. VAL = typical public validation reproductions.
# We only compute validation ourselves (GLUE test is hidden).
REFERENCES = {
    ("bert-base-uncased", "sst2"): {
        "paper_test": ("Devlin et al. 2019 test", 93.5),
        "typical_val": ("HF run_glue / common dev", 92.7),
        "lora_val": ("AutoPEFT LoRA BERT-base dev", 92.06),
    },
    ("bert-base-uncased", "rte"): {
        "paper_test": ("Devlin et al. 2019 test", 66.4),
        "typical_val": ("HF JeremiahZ bert-base-rte dev", 68.9),
        "lora_val": ("AutoPEFT LoRA BERT-base dev", 65.85),
    },
    ("roberta-base", "sst2"): {
        "paper_test": ("Liu et al. 2019 test-ish / leaderboard", 96.4),
        "typical_val": ("Liu et al. 2019 RoBERTa-base dev", 94.8),
        "lora_val": ("common LoRA reproductions ~dev", 94.0),
    },
    ("roberta-base", "rte"): {
        "paper_test": ("Liu et al. 2019 leaderboard-ish", 86.6),
        "typical_val": ("Liu et al. 2019 RoBERTa-base dev", 78.7),
        "lora_val": ("common HF reproductions dev", 72.2),
    },
}

FREEZE_ORDER = ["head_only", "last2", "full_ft"]
FREEZE_LABEL = {
    "head_only": "head only",
    "last2": "last-2 + head",
    "full_ft": "full fine-tune",
}

PRESETS = {
    "smoke": dict(
        models=("bert-base-uncased",),
        tasks=("rte",),
        methods=("head_only", "full_ft"),
        epochs_sst2=1,
        epochs_rte=2,
        sst2_cap=512,
        rte_cap=None,
        max_length=64,
    ),
    "quick": dict(
        models=("bert-base-uncased", "roberta-base"),
        tasks=("sst2", "rte"),
        methods=("head_only", "last2", "full_ft"),
        epochs_sst2=3,
        epochs_rte=8,
        sst2_cap=8000,
        rte_cap=None,
        max_length=128,
    ),
    "full": dict(
        models=("bert-base-uncased", "roberta-base"),
        tasks=("sst2", "rte"),
        methods=("head_only", "last2", "full_ft"),
        epochs_sst2=3,
        epochs_rte=10,
        sst2_cap=None,
        rte_cap=None,
        max_length=128,
    ),
}


# ---------------------------------------------------------------------------
# Small helpers
# ---------------------------------------------------------------------------

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_device() -> torch.device:
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        print(
            f"[device] {torch.cuda.get_device_name(0)}  "
            f"{props.total_memory/1024**3:.1f} GB",
            flush=True,
        )
        return torch.device("cuda")
    print("[device] CPU — this will be slow", flush=True)
    return torch.device("cpu")


def metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    y_true = y_true.astype(int)
    y_pred = y_pred.astype(int)
    out = {"accuracy": float(accuracy_score(y_true, y_pred))}
    try:
        out["f1"] = float(f1_score(y_true, y_pred, average="binary"))
    except Exception:
        out["f1"] = float("nan")
    try:
        out["mcc"] = float(matthews_corrcoef(y_true, y_pred))
    except Exception:
        out["mcc"] = float("nan")
    return out


def nested_attr(obj, path: tuple[str, ...]):
    cur = obj
    for name in path:
        cur = getattr(cur, name)
    return cur


def encoder_blocks(model, model_name: str):
    return nested_attr(model, MODELS[model_name]["encoder_attr"])


def apply_freeze(model, model_name: str, mode: str) -> None:
    if mode == "full_ft":
        for p in model.parameters():
            p.requires_grad = True
        return
    for p in model.parameters():
        p.requires_grad = False
    # classification head (BERT: classifier; RoBERTa: classifier dense+out_proj)
    if hasattr(model, "classifier"):
        for p in model.classifier.parameters():
            p.requires_grad = True
    if mode == "head_only":
        return
    if mode == "last2":
        blocks = encoder_blocks(model, model_name)
        for block in blocks[-2:]:
            for p in block.parameters():
                p.requires_grad = True
        pooler_path = MODELS[model_name]["pooler_attr"]
        if pooler_path is not None:
            try:
                pooler = nested_attr(model, pooler_path)
                for p in pooler.parameters():
                    p.requires_grad = True
            except AttributeError:
                pass
        return
    raise ValueError(mode)


def count_trainable(model) -> tuple[int, int]:
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    tot = sum(p.numel() for p in model.parameters())
    return tr, tot


# ---------------------------------------------------------------------------
# Data
# ---------------------------------------------------------------------------

def load_split(task: str, tokenizer, max_len: int, split: str, cap: int | None):
    from datasets import load_dataset

    raw = load_dataset("glue", task, split=split)
    if cap is not None:
        raw = raw.select(range(min(cap, len(raw))))
    meta = TASKS[task]
    s1, s2 = meta["s1"], meta["s2"]

    def tok(batch):
        if s2 is None:
            enc = tokenizer(
                batch[s1], truncation=True, padding="max_length", max_length=max_len
            )
        else:
            enc = tokenizer(
                batch[s1],
                batch[s2],
                truncation=True,
                padding="max_length",
                max_length=max_len,
            )
        enc["labels"] = batch["label"]
        return enc

    cols = raw.column_names
    ds = raw.map(tok, batched=True, remove_columns=cols)
    # RoBERTa has no token_type_ids; BERT does. Never request them.
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    return ds


# ---------------------------------------------------------------------------
# Train / eval
# ---------------------------------------------------------------------------

@torch.no_grad()
def eval_model(model, loader, device) -> tuple[float, np.ndarray, np.ndarray]:
    model.eval()
    losses, preds, labels = [], [], []
    for batch in loader:
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        y = batch["labels"].to(device)
        out = model(input_ids=ids, attention_mask=mask, labels=y)
        losses.append(out.loss.item() * y.size(0))
        preds.append(out.logits.argmax(-1).cpu().numpy())
        labels.append(y.cpu().numpy())
    pred = np.concatenate(preds)
    lab = np.concatenate(labels)
    loss = float(sum(losses) / max(len(lab), 1))
    return loss, lab, pred


def sharpness_gap(model, loader, device, rho: float = 0.02, max_batches: int = 4) -> float:
    """Random-direction loss gap on trainable weights. Restores params."""
    model.eval()
    trainable = [p for p in model.parameters() if p.requires_grad]
    if not trainable:
        return float("nan")

    def mean_loss() -> float:
        vals, n = 0.0, 0
        with torch.no_grad():
            for i, batch in enumerate(loader):
                if i >= max_batches:
                    break
                ids = batch["input_ids"].to(device)
                mask = batch["attention_mask"].to(device)
                y = batch["labels"].to(device)
                vals += model(input_ids=ids, attention_mask=mask, labels=y).loss.item()
                n += 1
        return vals / max(n, 1)

    base = mean_loss()
    noise = [torch.randn_like(p, dtype=torch.float32) for p in trainable]
    denom = math.sqrt(sum(float(g.square().sum().item()) for g in noise)) + 1e-12
    with torch.no_grad():
        for p, g in zip(trainable, noise):
            p.add_((rho * g / denom).to(device=p.device, dtype=p.dtype))
    pert = mean_loss()
    with torch.no_grad():
        for p, g in zip(trainable, noise):
            p.sub_((rho * g / denom).to(device=p.device, dtype=p.dtype))
    return float(pert - base)


def run_one(
    task: str,
    model_name: str,
    method: str,
    epochs: int,
    train_cap: int | None,
    batch_size: int,
    lr: float,
    weight_decay: float,
    max_length: int,
    seed: int,
    device: torch.device,
    no_fp16: bool,
    rho: float,
) -> dict[str, Any]:
    seed_everything(seed)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    tr_ds = load_split(task, tokenizer, max_length, "train", train_cap)
    va_ds = load_split(task, tokenizer, max_length, "validation", None)
    tr_dl = DataLoader(tr_ds, batch_size=batch_size, shuffle=True)
    va_dl = DataLoader(va_ds, batch_size=batch_size * 2, shuffle=False)
    probe_dl = DataLoader(tr_ds, batch_size=batch_size, shuffle=False)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    if getattr(model.config, "pad_token_id", None) is None:
        model.config.pad_token_id = tokenizer.pad_token_id
    apply_freeze(model, model_name, method)
    model.to(device)
    n_tr, n_tot = count_trainable(model)
    print(f"    trainable {n_tr:,} / {n_tot:,}", flush=True)

    opt = torch.optim.AdamW(
        (p for p in model.parameters() if p.requires_grad),
        lr=lr,
        weight_decay=weight_decay,
    )
    use_amp = device.type == "cuda" and not no_fp16
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    epoch_val: list[float] = []
    t0 = time.time()
    for ep in range(epochs):
        model.train()
        running, nseen = 0.0, 0
        for batch in tr_dl:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            y = batch["labels"].to(device)
            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=use_amp):
                loss = model(input_ids=ids, attention_mask=mask, labels=y).loss
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(
                (p for p in model.parameters() if p.requires_grad), 1.0
            )
            scaler.step(opt)
            scaler.update()
            running += loss.item() * y.size(0)
            nseen += y.size(0)
        va_loss_ep, yva_ep, pva_ep = eval_model(model, va_dl, device)
        acc_ep = float((yva_ep == pva_ep).mean())
        epoch_val.append(acc_ep)
        print(
            f"    epoch {ep+1:02d}/{epochs}  train_loss={running/max(nseen,1):.4f}  "
            f"val_acc={100*acc_ep:.2f}",
            flush=True,
        )

    tr_loss, ytr, ptr = eval_model(model, probe_dl, device)
    va_loss, yva, pva = eval_model(model, va_dl, device)
    tr_m = metrics(ytr, ptr)
    va_m = metrics(yva, pva)
    primary = TASKS[task]["metric"]
    sharp = sharpness_gap(model, probe_dl, device, rho=rho)

    row = {
        "task": task,
        "model_name": model_name,
        "method": method,
        "seed": seed,
        "epochs": epochs,
        "n_train": len(tr_ds),
        "n_val": len(va_ds),
        "trainable_params": n_tr,
        "total_params": n_tot,
        "train_loss": tr_loss,
        "train_acc": tr_m["accuracy"],
        "val_loss": va_loss,
        "val_accuracy": va_m["accuracy"],
        "val_f1": va_m["f1"],
        "val_mcc": va_m["mcc"],
        "val_primary": va_m[primary],
        "val_primary_metric": primary,
        "val_primary_pct": 100.0 * va_m[primary],
        "sharpness_gap": sharp,
        "interpolated": bool(tr_m["accuracy"] >= 0.99),
        "epoch_val": epoch_val,
        "seconds": time.time() - t0,
        "notes": "sst2 subsampled" if (task == "sst2" and train_cap) else "",
    }

    del model, opt, scaler
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return row


# ---------------------------------------------------------------------------
# Figures + report
# ---------------------------------------------------------------------------

def _mpl_style() -> None:
    plt.rcParams.update(
        {
            "font.size": 10,
            "axes.titlesize": 11,
            "axes.labelsize": 10,
            "legend.fontsize": 8,
            "figure.dpi": 140,
            "savefig.dpi": 300,
            "axes.spines.top": False,
            "axes.spines.right": False,
            "axes.grid": True,
            "grid.alpha": 0.28,
            "font.family": "DejaVu Sans",
        }
    )


METHOD_X = {m: i for i, m in enumerate(FREEZE_ORDER)}


def save_figures(rows: list[dict], out: Path) -> list[Path]:
    _mpl_style()
    paths: list[Path] = []
    tasks = [t for t in TASKS if any(r["task"] == t for r in rows)]
    models = [m for m in MODELS if any(r["model_name"] == m for r in rows)]

    # Fig 1 — val metric + published hline, by freeze depth
    fig, axes = plt.subplots(1, len(tasks), figsize=(5.3 * max(len(tasks), 1), 4.0), squeeze=False)
    for ax, task in zip(axes[0], tasks):
        for model_name in models:
            sub = sorted(
                [r for r in rows if r["task"] == task and r["model_name"] == model_name],
                key=lambda r: METHOD_X.get(r["method"], 99),
            )
            if not sub:
                continue
            xs = [METHOD_X[r["method"]] for r in sub]
            ys = [100.0 * r["val_primary"] for r in sub]
            ax.plot(
                xs,
                ys,
                marker="o",
                lw=2.0,
                color=MODELS[model_name]["color"],
                label=MODELS[model_name]["display"],
            )
            ref = REFERENCES.get((model_name, task), {})
            if "typical_val" in ref:
                ax.axhline(
                    ref["typical_val"][1],
                    color=MODELS[model_name]["color"],
                    ls=":",
                    lw=1.1,
                    alpha=0.85,
                )
        ax.set_xticks(range(len(FREEZE_ORDER)))
        ax.set_xticklabels([FREEZE_LABEL[m] for m in FREEZE_ORDER], rotation=18, ha="right")
        ax.set_title(TASKS[task]["display"])
        ax.set_ylabel(f"Validation {TASKS[task]['metric_label']} (%)")
        if task == tasks[0]:
            ax.legend(frameon=False, loc="best")
    fig.suptitle(
        "Freeze-depth sweep  ·  dotted = strongest published validation number",
        y=1.03,
        fontsize=12,
    )
    fig.tight_layout()
    p = out / "fig1_val_vs_freeze.png"
    fig.savefig(p, bbox_inches="tight")
    plt.close(fig)
    paths.append(p)

    # Fig 2 — interpolation proxy vs freeze
    fig, axes = plt.subplots(1, len(tasks), figsize=(5.3 * max(len(tasks), 1), 3.8), squeeze=False)
    for ax, task in zip(axes[0], tasks):
        for model_name in models:
            sub = sorted(
                [r for r in rows if r["task"] == task and r["model_name"] == model_name],
                key=lambda r: METHOD_X.get(r["method"], 99),
            )
            if not sub:
                continue
            ax.plot(
                [METHOD_X[r["method"]] for r in sub],
                [100.0 * r["train_acc"] for r in sub],
                marker="s",
                lw=2.0,
                color=MODELS[model_name]["color"],
                label=MODELS[model_name]["display"],
            )
        ax.axhline(99.0, color="0.5", ls=":", lw=0.9)
        ax.set_xticks(range(len(FREEZE_ORDER)))
        ax.set_xticklabels([FREEZE_LABEL[m] for m in FREEZE_ORDER], rotation=18, ha="right")
        ax.set_ylim(50, 102)
        ax.set_title(TASKS[task]["display"])
        ax.set_ylabel("Train accuracy (%)")
        if task == tasks[0]:
            ax.legend(frameon=False)
    fig.suptitle("Interpolation proxy  ·  dotted = 99% train acc", y=1.03, fontsize=12)
    fig.tight_layout()
    p = out / "fig2_train_acc.png"
    fig.savefig(p, bbox_inches="tight")
    plt.close(fig)
    paths.append(p)

    # Fig 3 — sharpness
    fig, axes = plt.subplots(1, len(tasks), figsize=(5.3 * max(len(tasks), 1), 3.8), squeeze=False)
    for ax, task in zip(axes[0], tasks):
        for model_name in models:
            sub = sorted(
                [r for r in rows if r["task"] == task and r["model_name"] == model_name],
                key=lambda r: METHOD_X.get(r["method"], 99),
            )
            if not sub:
                continue
            ax.plot(
                [METHOD_X[r["method"]] for r in sub],
                [r["sharpness_gap"] for r in sub],
                marker="^",
                lw=2.0,
                color=MODELS[model_name]["color"],
                label=MODELS[model_name]["display"],
            )
        ax.set_xticks(range(len(FREEZE_ORDER)))
        ax.set_xticklabels([FREEZE_LABEL[m] for m in FREEZE_ORDER], rotation=18, ha="right")
        ax.set_title(TASKS[task]["display"])
        ax.set_ylabel(r"Random-direction gap  $\Delta\mathcal{L}(\rho=0.02)$")
        if task == tasks[0]:
            ax.legend(frameon=False)
    fig.suptitle("Cheap sharpness probe on trainable weights", y=1.03, fontsize=12)
    fig.tight_layout()
    p = out / "fig3_sharpness.png"
    fig.savefig(p, bbox_inches="tight")
    plt.close(fig)
    paths.append(p)

    # Fig 4 — delta vs published validation
    labels, deltas, colors = [], [], []
    for r in sorted(rows, key=lambda z: (z["task"], z["model_name"], METHOD_X.get(z["method"], 99))):
        ref = REFERENCES.get((r["model_name"], r["task"]), {})
        if "typical_val" not in ref:
            continue
        labels.append(
            f"{r['task']}:{MODELS[r['model_name']]['display'][:4]}:{r['method']}"
        )
        deltas.append(100.0 * r["val_primary"] - ref["typical_val"][1])
        colors.append(MODELS[r["model_name"]]["color"])
    if labels:
        fig, ax = plt.subplots(figsize=(max(7.5, 0.42 * len(labels)), 4.0))
        ax.bar(range(len(labels)), deltas, color=colors, edgecolor="black", linewidth=0.35)
        ax.axhline(0.0, color="black", lw=1.0)
        ax.axhline(-2.0, color="0.5", ls="--", lw=0.8)
        ax.set_xticks(range(len(labels)))
        ax.set_xticklabels(labels, rotation=70, ha="right", fontsize=7)
        ax.set_ylabel("Δ vs published validation baseline (pp)")
        ax.set_title("Reproduction floor  ·  dashed = −2 pp (protocol-sane band)")
        fig.tight_layout()
        p = out / "fig4_delta_vs_published.png"
        fig.savefig(p, bbox_inches="tight")
        plt.close(fig)
        paths.append(p)

    # Fig 5 — complexity scatter
    fig, axes = plt.subplots(1, len(tasks), figsize=(5.3 * max(len(tasks), 1), 3.9), squeeze=False)
    markers = {"head_only": "o", "last2": "s", "full_ft": "D"}
    for ax, task in zip(axes[0], tasks):
        for model_name in models:
            sub = [r for r in rows if r["task"] == task and r["model_name"] == model_name]
            for r in sub:
                ax.scatter(
                    r["trainable_params"] / 1e6,
                    100.0 * r["val_primary"],
                    c=MODELS[model_name]["color"],
                    marker=markers.get(r["method"], "o"),
                    s=70,
                    zorder=3,
                    label=f"{MODELS[model_name]['display']} {r['method']}",
                )
                ax.annotate(
                    r["method"],
                    (r["trainable_params"] / 1e6, 100.0 * r["val_primary"]),
                    textcoords="offset points",
                    xytext=(5, 3),
                    fontsize=6,
                    color="0.25",
                )
        ax.set_xlabel("Trainable parameters (M)")
        ax.set_ylabel(f"Validation {TASKS[task]['metric_label']} (%)")
        ax.set_title(TASKS[task]["display"])
    fig.suptitle("Effective fine-tune complexity (freeze depth)", y=1.03, fontsize=12)
    fig.tight_layout()
    p = out / "fig5_complexity.png"
    fig.savefig(p, bbox_inches="tight")
    plt.close(fig)
    paths.append(p)

    # Fig 6 — RTE epoch traces
    rte = [r for r in rows if r["task"] == "rte" and r.get("epoch_val")]
    if rte:
        fig, ax = plt.subplots(figsize=(7.2, 4.0))
        for r in sorted(rte, key=lambda z: (z["model_name"], METHOD_X.get(z["method"], 99))):
            ax.plot(
                range(1, len(r["epoch_val"]) + 1),
                [100.0 * v for v in r["epoch_val"]],
                lw=1.7,
                color=MODELS[r["model_name"]]["color"],
                ls={"head_only": ":", "last2": "--", "full_ft": "-"}[r["method"]],
                marker="o",
                ms=3.5,
                label=f"{MODELS[r['model_name']]['display']} / {FREEZE_LABEL[r['method']]}",
            )
        ax.set_xlabel("Epoch")
        ax.set_ylabel("RTE validation accuracy (%)")
        ax.set_title("Epoch-wise traces on RTE (n=2,490; interpolation reachable)")
        ax.legend(frameon=False, fontsize=7, ncol=2)
        fig.tight_layout()
        p = out / "fig6_rte_epochs.png"
        fig.savefig(p, bbox_inches="tight")
        plt.close(fig)
        paths.append(p)

    return paths


def write_report(rows: list[dict], out: Path) -> Path:
    pd.DataFrame(
        [{k: v for k, v in r.items() if k != "epoch_val"} for r in rows]
    ).to_csv(out / "results.csv", index=False)
    (out / "results.json").write_text(json.dumps(rows, indent=2))

    lines = [
        "# GLUE freeze-depth sanity check",
        "",
        "Validation only (GLUE test is hidden). Deltas use the published **validation** number.",
        "QUICK mode trains SST-2 on 8k examples — treat SST-2 as a protocol check, RTE as the DD probe.",
        "",
        "| Task | Model | Freeze | Trainable | Train acc | Val | Published val | Δ pp | Paper test | Sharpness | Interp. | Time |",
        "|---|---|---|---:|---:|---:|---:|---:|---:|---:|---|---:|",
    ]
    for r in sorted(rows, key=lambda z: (z["task"], z["model_name"], METHOD_X.get(z["method"], 99))):
        ref = REFERENCES.get((r["model_name"], r["task"]), {})
        val_pub = ref.get("typical_val", ("", float("nan")))[1]
        paper = ref.get("paper_test", ("", float("nan")))[1]
        our = 100.0 * r["val_primary"]
        delta = our - val_pub if val_pub == val_pub else float("nan")
        lines.append(
            f"| {r['task']} | {MODELS[r['model_name']]['display']} | {r['method']} | "
            f"{r['trainable_params']/1e6:.2f}M | {100*r['train_acc']:.1f} | {our:.1f} | "
            f"{val_pub:.1f} | {delta:+.1f} | {paper:.1f} | {r['sharpness_gap']:.4f} | "
            f"{r['interpolated']} | {r['seconds']:.0f}s |"
        )
    lines += [
        "",
        "## How to read this",
        "",
        "1. **Reproduction floor.** `full_ft` on RTE should land within ~3 pp of the published "
        "validation number (BERT ~69, RoBERTa ~79). SST-2 in QUICK is subsampled, so −1 to −4 pp "
        "vs 92.7 / 94.8 is not a bug.",
        "2. **Signature of the idea.** head_only underfits. last2 reaches high train acc, "
        "possibly worse val and higher sharpness (ridge). full_ft recovers val and drops "
        "sharpness (slack past the ridge). That is freeze-wise double descent, not a GLUE SOTA claim.",
        "3. **Single seed, RTE val = 277.** ±2–3 pp is noise. Do not over-read a 1 pp wiggle.",
        "4. **Beating RoBERTa-base full-FT published 78.7 / 94.8 is not the bar.**",
        "",
    ]
    path = out / "summary.md"
    path.write_text("\n".join(lines))
    print("\n" + "\n".join(lines), flush=True)
    return path


def print_verdict(rows: list[dict]) -> None:
    print("=" * 72)
    print("SANITY VERDICT")
    print("=" * 72)
    for task in [t for t in TASKS if any(r["task"] == t for r in rows)]:
        block = [r for r in rows if r["task"] == task]
        print(f"\n[{TASKS[task]['display']}]")
        for model_name in [m for m in MODELS if any(r["model_name"] == m for r in block)]:
            sub = [r for r in block if r["model_name"] == model_name]
            best = max(sub, key=lambda r: r["val_primary"])
            ref = REFERENCES.get((model_name, task), {})
            pub = ref.get("typical_val", ("?", float("nan")))
            d = 100.0 * best["val_primary"] - pub[1]
            band = "IN BAND" if d >= -3.0 else "BELOW BAND"
            print(
                f"  {MODELS[model_name]['display']}: best={best['method']}  "
                f"val={100*best['val_primary']:.1f}  vs {pub[1]:.1f} ({pub[0]})  "
                f"Δ {d:+.1f} pp  [{band}]"
            )
            by_m = {r["method"]: r for r in sub}
            if "last2" in by_m and "full_ft" in by_m:
                a, b = by_m["last2"], by_m["full_ft"]
                if a["train_acc"] >= 0.97 and b["val_primary"] > a["val_primary"] + 0.005:
                    print(
                        "    pulse: last2 interpolates and full_ft is better on val "
                        f"({100*a['val_primary']:.1f} → {100*b['val_primary']:.1f})."
                    )
                if a["sharpness_gap"] > b["sharpness_gap"] + 1e-4:
                    print(
                        f"    landscape: last2 sharper ({a['sharpness_gap']:.4f}) than "
                        f"full_ft ({b['sharpness_gap']:.4f})."
                    )
    print(
        "\nValue-add is a freeze-wise ridge (high train acc, worse val, high sharpness) "
        "that full_ft walks off. Matching Devlin test 93.5 / 66.4 is not required."
    )
    print("=" * 72)


# ---------------------------------------------------------------------------
# CLI
# ---------------------------------------------------------------------------

def parse_args(argv: list[str] | None = None) -> argparse.Namespace:
    p = argparse.ArgumentParser()
    p.add_argument("--preset", choices=sorted(PRESETS), default="quick")
    p.add_argument("--out", default="glue_dd_results")
    p.add_argument("--models", nargs="*", default=None)
    p.add_argument("--tasks", nargs="*", default=None)
    p.add_argument("--methods", nargs="*", default=None)
    p.add_argument("--batch-size", type=int, default=16)
    p.add_argument("--lr", type=float, default=2e-5)
    p.add_argument("--weight-decay", type=float, default=0.01)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--rho", type=float, default=0.02)
    p.add_argument("--no-fp16", action="store_true")
    if argv is None:
        argv = sys.argv[1:]
    args, unknown = p.parse_known_args(argv)
    if unknown:
        print(f"[args] ignoring {unknown}", flush=True)
    return args


def main(argv: list[str] | None = None) -> None:
    warnings.filterwarnings("ignore")
    args = parse_args(argv)
    preset = PRESETS[args.preset]
    models = tuple(args.models) if args.models else preset["models"]
    tasks = tuple(args.tasks) if args.tasks else preset["tasks"]
    methods = tuple(args.methods) if args.methods else preset["methods"]

    out = Path(args.out)
    if Path("/kaggle/working").exists() and args.out == "glue_dd_results":
        out = Path("/kaggle/working/glue_dd_results")
    out.mkdir(parents=True, exist_ok=True)

    device = get_device()
    seed_everything(args.seed)
    (out / "config.json").write_text(
        json.dumps(
            {
                "preset": args.preset,
                "models": models,
                "tasks": tasks,
                "methods": methods,
                "lr": args.lr,
                "batch_size": args.batch_size,
                "seed": args.seed,
                "epochs": {"sst2": preset["epochs_sst2"], "rte": preset["epochs_rte"]},
                "sst2_cap": preset["sst2_cap"],
            },
            indent=2,
        )
    )

    combo = [(t, m, f) for t in tasks for m in models for f in methods]
    print(f"[cfg] preset={args.preset}  runs={len(combo)}  out={out}", flush=True)

    rows: list[dict] = []
    for i, (task, model_name, method) in enumerate(combo, 1):
        epochs = preset["epochs_sst2"] if task == "sst2" else preset["epochs_rte"]
        cap = preset["sst2_cap"] if task == "sst2" else preset["rte_cap"]
        print(
            f"\n[{i}/{len(combo)}] {task} | {model_name} | {method} | "
            f"epochs={epochs} | cap={cap}",
            flush=True,
        )
        try:
            row = run_one(
                task=task,
                model_name=model_name,
                method=method,
                epochs=epochs,
                train_cap=cap,
                batch_size=args.batch_size,
                lr=args.lr,
                weight_decay=args.weight_decay,
                max_length=preset["max_length"],
                seed=args.seed,
                device=device,
                no_fp16=args.no_fp16,
                rho=args.rho,
            )
            rows.append(row)
            print(
                f"  -> val={row['val_primary_pct']:.1f}  train_acc={100*row['train_acc']:.1f}  "
                f"sharp={row['sharpness_gap']:.4f}  time={row['seconds']:.0f}s",
                flush=True,
            )
            (out / "results.json").write_text(json.dumps(rows, indent=2))
        except Exception as exc:
            print(f"  FAILED: {type(exc).__name__}: {exc}", flush=True)
            rows.append(
                {
                    "task": task,
                    "model_name": model_name,
                    "method": method,
                    "error": f"{type(exc).__name__}: {exc}",
                    "val_primary": float("nan"),
                    "val_primary_pct": float("nan"),
                    "train_acc": float("nan"),
                    "sharpness_gap": float("nan"),
                    "trainable_params": 0,
                    "total_params": 0,
                    "interpolated": False,
                    "seconds": 0.0,
                    "epoch_val": [],
                    "n_train": 0,
                    "n_val": 0,
                    "val_f1": float("nan"),
                    "val_mcc": float("nan"),
                    "val_accuracy": float("nan"),
                    "val_primary_metric": TASKS[task]["metric"],
                }
            )

    good = [r for r in rows if "error" not in r]
    if good:
        write_report(good, out)
        figs = save_figures(good, out)
        print_verdict(good)
        print("[wrote]")
        for p in [out / "results.csv", out / "summary.md", *figs]:
            print(f"  {p}")
    else:
        print("[fatal] every run failed", flush=True)
        sys.exit(1)


if __name__ == "__main__":
    main()

[args] ignoring ['--f=/Users/sagarkumaracharya/Library/Jupyter/runtime/kernel-v2-349025I71bAonS70R.json']
[device] CPU — this will be slow
[cfg] preset=quick  runs=12  out=glue_dd_results

[1/12] sst2 | bert-base-uncased | head_only | epochs=3 | cap=8000


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

  FAILED: RemoteProtocolError: Server disconnected without sending a response.

[2/12] sst2 | bert-base-uncased | last2 | epochs=3 | cap=8000


KeyboardInterrupt: 